# GroupDNA - WhatsApp Group Chat Analyzer

Name: Vrinda K M

Batch: DataScience November Batch

### Set up

In [2]:
import numpy as np
from datetime import datetime, timedelta

with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
    content = f.read()

lines = content.split('\n')
print('Total lines:', len(lines))

Total lines: 3178


### Feature 1: Parser

In [3]:
def starts_with_date(line):
    if len(line) < 8:
        return False
    return line[0:2].isdigit() and line[2] == '/' and line[3:5].isdigit() and line[5] == '/' and line[6:8].isdigit()


def get_time(stamp):
    # stamp looks like '12/04/24, 23:14'
    date_part, time_part = stamp.split(', ')
    day, month, year = date_part.split('/')
    hour, minute = time_part.split(':')
    return datetime(2000 + int(year), int(month), int(day), int(hour), int(minute))


months = ['January', 'February', 'March', 'April', 'May', 'June', 'July',
          'August', 'September', 'October', 'November', 'December']


def show_date(d):
    # 2024-04-01 -> '01 April 2024'
    return str(d.day).zfill(2) + ' ' + months[d.month - 1] + ' ' + str(d.year)


def short_date(d):
    # 2024-04-16 -> '16 Apr'
    return str(d.day).zfill(2) + ' ' + months[d.month - 1][:3]


def parse_chat(lines):
    messages = []
    system = 0
    media = 0
    deleted = 0
    group_name = ''

    for line in lines:
        line = line.strip()
        if line == '':
            continue

        # no date at start means it is the next line of the previous message
        if not starts_with_date(line):
            if len(messages) > 0:
                messages[-1]['text'] += ' ' + line
            continue

        stamp, rest = line.split(' - ', 1)

        # system messages have no "name: " part
        if ': ' not in rest:
            system += 1
            if 'created group' in rest:
                group_name = rest.split('"')[1]
            continue

        sender, text = rest.split(': ', 1)
        dt = get_time(stamp)

        if text == '<Media omitted>':
            kind = 'media'
            media += 1
        elif text == 'This message was deleted':
            kind = 'deleted'
            deleted += 1
        else:
            kind = 'text'

        messages.append({'timestamp': stamp, 'time': dt, 'date': dt.date(), 'hour': dt.hour,
                         'sender': sender, 'text': text, 'kind': kind})

    info = {'system': system, 'media': media, 'deleted': deleted, 'group': group_name}
    return messages, info


messages, info = parse_chat(lines)
people = set()
for m in messages:
    people.add(m['sender'])

first_day = messages[0]['date']
last_day = messages[-1]['date']
total_days = (last_day - first_day).days + 1

print(f"Successfully parsed {len(messages)} messages from {len(people)} participants over {total_days} days, "
      f"skipped {info['system']} system messages, {info['media']} media-omitted, {info['deleted']} deleted messages.")

print('\nFirst 5:')
for m in messages[:5]:
    print(m['timestamp'], m['sender'], ':', m['text'])
print('\nLast 5:')
for m in messages[-5:]:
    print(m['timestamp'], m['sender'], ':', m['text'])

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.

First 5:
01/04/24, 01:17 Rahul : scene fix
01/04/24, 01:17 Rahul : haan
01/04/24, 01:18 Rahul : kya scene
01/04/24, 02:13 Rahul : abhi free hai?
01/04/24, 02:13 Rahul : abey

Last 5:
30/05/24, 19:14 Priya : Take care everyone
30/05/24, 19:28 Priya : Karan that sounds tough, take care
30/05/24, 21:17 Aman : the existential dread is back
30/05/24, 21:30 Karan : Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around campus collecting signatures for the project document, ate lunch at four PM standing in the corridor, came back to hostel and realized I forgot my charger in the lab.
30/05/24, 23:31 Aman : anyone awake?


### Feature 2: Group Overview

In [4]:
count = {}
for m in messages:
    count[m['sender']] = count.get(m['sender'], 0) + 1

ranking = sorted(count, key=lambda p: count[p], reverse=True)
total = len(messages)


def make_bar(value, biggest, size=25):
    n = round(value / biggest * size)
    if n == 0:
        return '.'
    return '█' * n


# words per person (only real text messages)
words_sent = {}
text_count = {}
media_count = {}
deleted_count = {}
for p in people:
    words_sent[p] = 0
    text_count[p] = 0
    media_count[p] = 0
    deleted_count[p] = 0

for m in messages:
    p = m['sender']
    if m['kind'] == 'text':
        text_count[p] += 1
        words_sent[p] += len(m['text'].split())
    elif m['kind'] == 'media':
        media_count[p] += 1
    else:
        deleted_count[p] += 1

avg_words = {}
for p in people:
    if text_count[p] > 0:
        avg_words[p] = words_sent[p] / text_count[p]
    else:
        avg_words[p] = 0

print('=' * 60)
print('  GROUP OVERVIEW')
print('=' * 60)
print(f"  {'Group':<16}: {info['group']}")
print(f"  {'Period':<16}: {show_date(first_day)} to {show_date(last_day)} ({total_days} days)")
print(f"  {'Total messages':<16}: {total:,}")
print(f"  {'Participants':<16}: {len(people)}")
print()
print('  MESSAGES PER PERSON')
for p in ranking:
    print(f"    {p:<8}: {count[p]:>4} ({count[p] / total * 100:4.1f}%)   avg {avg_words[p]:.1f} words, "
          f"{media_count[p]} media, {deleted_count[p]} deleted")

  GROUP OVERVIEW
  Group           : Hostel Bois 4ever
  Period          : 01 April 2024 to 30 May 2024 (60 days)
  Total messages  : 3,174
  Participants    : 6

  MESSAGES PER PERSON
    Rahul   :  953 (30.0%)   avg 2.6 words, 7 media, 6 deleted
    Priya   :  718 (22.6%)   avg 5.0 words, 4 media, 2 deleted
    Neha    :  635 (20.0%)   avg 5.3 words, 8 media, 3 deleted
    Aman    :  490 (15.4%)   avg 5.0 words, 4 media, 2 deleted
    Karan   :  354 (11.2%)   avg 57.0 words, 7 media, 2 deleted
    Vikas   :   24 ( 0.8%)   avg 1.8 words, 2 media, 0 deleted


### Feature 3: Most Active Day and Hour

In [5]:
day_count = {}
hour_count = {}
for m in messages:
    day_count[m['date']] = day_count.get(m['date'], 0) + 1
    hour_count[m['hour']] = hour_count.get(m['hour'], 0) + 1

busy_day = max(day_count, key=lambda d: day_count[d])
busy_hour = max(hour_count, key=lambda h: hour_count[h])
hour_text = f"{busy_hour:02d}:00 - {(busy_hour + 1) % 24:02d}:00"

print(f"  Busiest day  : {show_date(busy_day)} ({day_count[busy_day]} messages)")
print(f"  Busiest hour : {hour_text} (avg {hour_count[busy_hour] / total_days:.0f} messages per day)")

  Busiest day  : 04 May 2024 (76 messages)
  Busiest hour : 18:00 - 19:00 (avg 4 messages per day)


### Feature 4: Activity Heatmap (NumPy)

In [6]:
heatmap = np.zeros((len(ranking), 24), dtype=int)
for m in messages:
    row = ranking.index(m['sender'])
    heatmap[row, m['hour']] += 1

print('row totals:', heatmap.sum(axis=1))   # should match the counts above


def heatmap_lines(night_owl=''):
    out = ['  ACTIVITY HEATMAP (messages by hour)']
    out.append('           00    03    06    09    12    15    18    21')
    for i in range(len(ranking)):
        row = heatmap[i]
        biggest = row.max()
        s = ''
        for value in row:
            ratio = value / biggest
            if ratio <= 0.25:
                s += '. '
            elif ratio <= 0.5:
                s += '░ '
            elif ratio <= 0.75:
                s += '▒ '
            else:
                s += '█ '
        if ranking[i] == night_owl:
            s += ' <- NIGHT OWL'
        out.append(f'  {ranking[i]:<8} {s}')
    return out


for l in heatmap_lines():
    print(l)

row totals: [953 718 635 490 354  24]
  ACTIVITY HEATMAP (messages by hour)
           00    03    06    09    12    15    18    21
  Rahul    . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
  Priya    . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
  Neha     . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
  Aman     ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒ 
  Karan    . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
  Vikas    . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 


### Feature 5: Top Words

In [7]:
stop_words = ['i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'this',
              'that', 'you', 'me', 'my', 'we', 'are', 'was', 'be', 'at', 'with', 'so', 'but', 'if',
              'do', 'not', 'no', 'just', 'have', 'will', 'can', 'u', 'im', 'ok', 'all', 'what',
              'hai', 'h', 'ka', 'ki', 'ke', 'ko', 'se', 'toh', 'bhi', 'na', 'ho', 'tu', 'ye', 'wo',
              'aur', 'main', 'mai', 'hu', 'hain', 'tha', 'par', 'pe']
punct = '.,!?;:"\'()[]-_*/…'


def count_words(msgs):
    freq = {}
    for m in msgs:
        if m['kind'] != 'text':
            continue
        for w in m['text'].lower().split():
            w = w.strip(punct)
            if w == '' or w in stop_words or not w.isalpha():
                continue
            freq[w] = freq.get(w, 0) + 1
    return freq


word_freq = count_words(messages)
top_words = sorted(word_freq, key=lambda w: word_freq[w], reverse=True)[:10]

print("  THIS GROUP'S FAVOURITE WORDS")
for w in top_words:
    print(f'    {w:<9} {make_bar(word_freq[w], word_freq[top_words[0]]):<26} {word_freq[w]}')

print('\n  Top 5 words per person')
for p in ranking:
    f = count_words([m for m in messages if m['sender'] == p])
    top5 = sorted(f, key=lambda w: f[w], reverse=True)[:5]
    print(f'    {p:<8}:', ', '.join(top5))

  THIS GROUP'S FAVOURITE WORDS
    how       █████████████████████████  321
    guys      █████████████████████████  318
    about     █████████████████████      274
    am        ████████████████████       260
    today     ████████████████████       257
    he        █████████████████          220
    his       █████████████████          217
    which     ████████████████           202
    everyone  ███████████████            187
    telling   ██████████████             179

  Top 5 words per person
    Rahul   : bhai, scene, kya, yaar, ja
    Priya   : please, everyone, aman, anyone, okay
    Neha    : am, guys, cant, today, way
    Aman    : am, sleep, anyone, wonder, night
    Karan   : how, about, his, which, he
    Vikas   : haha, sorry, busy, haan, exam


### Feature 6: Response Speed and Silent Streaks

In [8]:
# response time = gap between someone else's message and this person's next message
gaps = {}
for i in range(1, len(messages)):
    prev = messages[i - 1]
    cur = messages[i]
    if cur['sender'] != prev['sender']:
        seconds = (cur['time'] - prev['time']).total_seconds()
        if cur['sender'] not in gaps:
            gaps[cur['sender']] = []
        gaps[cur['sender']].append(seconds)

avg_reply = {}
for p in gaps:
    avg_reply[p] = sum(gaps[p]) / len(gaps[p])


def nice_time(sec):
    if sec < 3600:
        return f'{sec / 60:.1f} minutes'
    return f'{sec / 3600:.1f} hours'


active_days = {}
for p in people:
    active_days[p] = set()
for m in messages:
    active_days[m['sender']].add(m['date'])

streak = {}
for p in people:
    best, run = 0, 0
    best_start, best_end, run_start = None, None, None
    day = first_day
    while day <= last_day:
        if day in active_days[p]:
            run = 0
        else:
            if run == 0:
                run_start = day
            run += 1
            if run > best:
                best = run
                best_start = run_start
                best_end = day
        day += timedelta(days=1)
    streak[p] = (best, best_start, best_end)


def response_lines():
    fast = min(avg_reply, key=lambda p: avg_reply[p])
    slow = max(avg_reply, key=lambda p: avg_reply[p])
    out = ['  RESPONSE PATTERNS',
           f'    Fastest replier : {fast:<8} (avg {nice_time(avg_reply[fast])})',
           f'    Slowest replier : {slow:<8} (avg {nice_time(avg_reply[slow])})',
           '',
           '  LONGEST SILENT STREAKS']
    for p in sorted(streak, key=lambda p: streak[p][0], reverse=True):
        days, start, end = streak[p]
        if days == 0:
            out.append(f'    {p:<8}: 0 days (never went silent)')
        elif days == 1:
            out.append(f'    {p:<8}: 1 day')
        else:
            out.append(f"    {p:<8}: {days} days ({short_date(start)} - {short_date(end)})")
    return out


for l in response_lines():
    print(l)

  RESPONSE PATTERNS
    Fastest replier : Rahul    (avg 34.9 minutes)
    Slowest replier : Aman     (avg 55.4 minutes)

  LONGEST SILENT STREAKS
    Vikas   : 11 days (23 Apr - 03 May)
    Aman    : 0 days (never went silent)
    Priya   : 0 days (never went silent)
    Karan   : 0 days (never went silent)
    Neha    : 0 days (never went silent)
    Rahul   : 0 days (never went silent)


### Feature 7: Personality Archetypes

In [9]:
caring_words = ['okay', 'safe', 'eat', 'sleep', 'please', 'reminder']
caring_phrases = ['take care', 'are you', 'drink water', "don't forget"]
laugh_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

# my own archetype: THE LAST-MINUTE MUGGU
# the person who keeps talking about exams, assignments, deadlines and proxy.
# I check what % of all their words are study words (not % of messages, because
# a person with long messages would get it just by writing more). more than 3% = muggu.
study_words = ['exam', 'exams', 'assignment', 'deadline', 'submit', 'viva', 'quiz', 'syllabus',
               'attendance', 'proxy', 'notes', 'backlog', 'midsem', 'endsem', 'marks', 'cgpa']

# bursts = how many messages a person sends in a row before someone else talks
bursts = {}
current, length = messages[0]['sender'], 0
for m in messages:
    if m['sender'] == current:
        length += 1
    else:
        bursts.setdefault(current, []).append(length)
        current, length = m['sender'], 1
bursts.setdefault(current, []).append(length)


def texts_of(p):
    return [m['text'] for m in messages if m['sender'] == p and m['kind'] == 'text']


def percent(p, check):
    texts = texts_of(p)
    if len(texts) == 0:
        return 0
    hits = 0
    for t in texts:
        if check(t):
            hits += 1
    return hits / len(texts)


def word_list(t):
    return [w.strip(punct) for w in t.lower().split()]


def is_caring(t):
    words = word_list(t)
    for w in caring_words:
        if w in words:
            return True
    for ph in caring_phrases:
        if ph in t.lower():
            return True
    return False


def is_drama(t):
    return (len(t) >= 3 and t.isupper()) or t.count('!') >= 2


def is_funny(t):
    for w in word_list(t):
        if w in laugh_words or w.startswith('haha'):
            return True
    return False


def spammer(p):
    return sum(bursts[p]) / len(bursts[p])

def group_mom(p):
    return percent(p, is_caring)

def night_owl(p):
    row = heatmap[ranking.index(p)]
    night = row[23] + row[0:5].sum()       # 23:00 to 04:59
    return night / row.sum()

def storyteller(p):
    return avg_words[p]

def drama_queen(p):
    return percent(p, is_drama)

def ghost(p):
    return 1 - len(active_days[p]) / total_days

def comedian(p):
    return percent(p, is_funny)

def question_master(p):
    return percent(p, lambda t: t.endswith('?'))

def muggu(p):
    words = []
    for t in texts_of(p):
        words += word_list(t)
    if len(words) == 0:
        return 0
    return sum(1 for w in words if w in study_words) / len(words)


# name: (function, threshold). score = value / threshold, so above 1 means they pass the rule
rules = {
    'THE SPAMMER': (spammer, 3),
    'THE GROUP MOM': (group_mom, 0.30),
    'THE NIGHT OWL': (night_owl, 0.60),
    'THE STORYTELLER': (storyteller, 30),
    'THE DRAMA QUEEN': (drama_queen, 0.30),
    'THE GHOST': (ghost, 0.60),
    'THE COMEDIAN': (comedian, 0.20),
    'THE QUESTION MASTER': (question_master, 0.25),
    'THE LAST-MINUTE MUGGU': (muggu, 0.03),
}

values = {}
scores = []
for p in ranking:
    values[p] = {}
    for name in rules:
        func, limit = rules[name]
        values[p][name] = func(p)
        scores.append((func(p) / limit, name, p))

# tie-breaking: highest score in the whole group is given out first. once a person or an
# archetype is taken it can't be used again, so if two people pass the same rule the one
# with the higher score gets it and the other one gets their next best archetype.
scores.sort(reverse=True)
archetype = {}
used = []
for score, name, p in scores:
    if p not in archetype and name not in used:
        archetype[p] = name
        used.append(name)


def reason(p):
    a = archetype[p]
    v = values[p][a]
    if a == 'THE SPAMMER':
        return f'avg {v:.1f} msgs in a row'
    if a == 'THE GROUP MOM':
        return f'{v * 100:.0f}% caring messages'
    if a == 'THE NIGHT OWL':
        return f'{v * 100:.1f}% msgs between 23h-04h'
    if a == 'THE STORYTELLER':
        return f'avg {v:.1f} words per msg'
    if a == 'THE DRAMA QUEEN':
        return f'{v * 100:.1f}% ALL-CAPS messages'
    if a == 'THE GHOST':
        return f'silent on {total_days - len(active_days[p])} of {total_days} days'
    if a == 'THE COMEDIAN':
        return f'{v * 100:.0f}% msgs with lol/haha'
    if a == 'THE QUESTION MASTER':
        return f'{v * 100:.0f}% msgs end with ?'
    return f'{v * 100:.1f}% of words about exams'


def archetype_lines():
    out = ['  PERSONALITY ARCHETYPES']
    for p in ranking:
        out.append(f'    {p:<8} → {archetype[p]:<22} ({reason(p)})')
    return out


for l in archetype_lines():
    print(l)

  PERSONALITY ARCHETYPES
    Rahul    → THE SPAMMER            (avg 4.5 msgs in a row)
    Priya    → THE GROUP MOM          (62% caring messages)
    Neha     → THE DRAMA QUEEN        (63.3% ALL-CAPS messages)
    Aman     → THE NIGHT OWL          (79.8% msgs between 23h-04h)
    Karan    → THE STORYTELLER        (avg 57.0 words per msg)
    Vikas    → THE LAST-MINUTE MUGGU  (5.0% of words about exams)


### Feature 8: Final Report

In [10]:
owl = ''
for p in archetype:
    if archetype[p] == 'THE NIGHT OWL':
        owl = p

print('=' * 60)
print(f'  GROUPDNA REPORT — "{info["group"]}"')
print(f'  {total_days} days • {total:,} messages • {len(people)} members')
print('=' * 60)
print()
print(f"  Period       : {show_date(first_day)} to {show_date(last_day)}")
print(f"  Busiest day  : {show_date(busy_day)} ({day_count[busy_day]} messages)")
print(f"  Busiest hour : {hour_text}")
print()
print('  MESSAGES PER PERSON')
for p in ranking:
    print(f'    {p:<8}{make_bar(count[p], count[ranking[0]]):<26}{count[p]:>4} ({count[p] / total * 100:4.1f}%)')
print()
for l in heatmap_lines(owl):
    print(l)
print()
print("  THIS GROUP'S FAVOURITE WORDS")
for w in top_words[:5]:
    print(f'    {w:<9} {make_bar(word_freq[w], word_freq[top_words[0]]):<26} {word_freq[w]}')
print()
for l in response_lines():
    print(l)
print()
for l in archetype_lines():
    print(l)
print()
print('=' * 60)
print('  Generated by GroupDNA • Built with Python + NumPy')
print('=' * 60)

  GROUPDNA REPORT — "Hostel Bois 4ever"
  60 days • 3,174 messages • 6 members

  Period       : 01 April 2024 to 30 May 2024
  Busiest day  : 04 May 2024 (76 messages)
  Busiest hour : 18:00 - 19:00

  MESSAGES PER PERSON
    Rahul   █████████████████████████  953 (30.0%)
    Priya   ███████████████████        718 (22.6%)
    Neha    █████████████████          635 (20.0%)
    Aman    █████████████              490 (15.4%)
    Karan   █████████                  354 (11.2%)
    Vikas   █                           24 ( 0.8%)

  ACTIVITY HEATMAP (messages by hour)
           00    03    06    09    12    15    18    21
  Rahul    . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
  Priya    . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
  Neha     . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
  Aman     ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒  <- NIGHT OWL
  Karan    . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
  Vikas    . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 

  

### Testing edge cases

In [11]:
# 1. empty file
msgs, inf = parse_chat([])
print('empty file ->', len(msgs), 'messages')

# 2. only one person
one = ['01/04/24, 10:00 - Rahul: hi', '01/04/24, 10:01 - Rahul: koi hai?', '02/04/24, 22:30 - Rahul: bhai']
msgs, inf = parse_chat(one)
print('one person ->', len(msgs), 'messages from', set(m['sender'] for m in msgs))

# 3. messy lines
messy = ['01/04/24, 01:13 - Priya created group "Test"',
         '',
         '12/04/24, 23:14 - Rahul: <Media omitted>',
         '12/04/24, 23:18 - Priya: please sleep',
         'this is the second line',
         '13/04/24, 09:15 - Neha: This message was deleted',
         '13/04/24, 09:16 - Neha: time: 9:16 AM']
msgs, inf = parse_chat(messy)
print('messy ->', len(msgs), 'messages', inf)
print(msgs[1]['text'])
print(msgs[3]['text'])

empty file -> 0 messages
one person -> 3 messages from {'Rahul'}
messy -> 4 messages {'system': 1, 'media': 1, 'deleted': 1, 'group': 'Test'}
please sleep this is the second line
time: 9:16 AM


### Reflection

Hardest part is the archetypes. Some people matched more than one archetype, so I had to give each one a score and pick the highest, making sure no two people got the same one.

What I would do differently is add more Hinglish stop words, because words like "hai" and "kya" kept showing up at the top.

